---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Homework 5**: RAG System for Fordham University

### 📅 **Due Date**: Day of Lecture 7, 11:59 PM

### Difficulty: ★★★★☆


**Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

In this homework, you'll complete the RAG (Retrieval Augmented Generation) system you started building in Lecture 5. You will build an end-to-end pipeline that can answer questions about Fordham University using real data scraped from the Fordham website.

This is an open-ended assignment — there is no single right implementation, and you're encouraged to experiment with chunking strategies, embedding models, prompt design, and retrieval parameters to improve your system. **I will grade your system by testing it with specific questions that I know the answer to.**

---

## Instructions

- You may use ChatGPT, Claude, documentation, Stack Overflow, etc. When using external resources, briefly cite them in a comment.
- Your submission must include **pre-computed embeddings** and any other artifacts needed so that I can run your RAG system without recomputing anything expensive. Share it with us in a way that makes sense.
- Run all cells before submitting to ensure they work.

**Submission:**
1. Create a branch called `homework-5`
2. Commit and push your work (notebook + Streamlit app + saved embeddings/artifacts in `temp/`)
3. Create a PR and merge to main
4. Submit the `.ipynb` file on Blackboard

---

## Step 1: Load and Chunk the Fordham Website Data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more. The first line of every file is the **URL** of the page it was scraped from. The rest is the page content in Markdown.

Think about: chunk size, what to split on (paragraphs, headers, fixed length, etc.), whether chunks should overlap, and how to track which page each chunk came from.

In [44]:
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import Counter
import string
from pathlib import Path
import litellm
# Import ONLY data loading from helpers
import sys
sys.path.append('../scripts')
from tqdm.auto import tqdm
import tiktoken
import asyncio
import textwrap


print("All imports successful!")

All imports successful!


In [45]:
#Load files into lists without extracting from zip

# Path to the zip archive
zip_path = "data/fordham-website.zip"

# List to hold documents
documents = []

# Open zip archive
with zipfile.ZipFile(zip_path, "r") as z:
    # Iterate over all markdown files
    for file_name in z.namelist():
        if file_name.endswith(".md"):
            # Read file content safely
            with z.open(file_name) as f:
                try:
                    content = f.read().decode("utf-8")
                except UnicodeDecodeError:
                    # fallback: replace invalid bytes
                    content = f.read().decode("utf-8", errors="replace")
            
            # Split first line as URL, rest as page content
            lines = content.splitlines()
            url = lines[0] if lines else ""
            page_content = "\n".join(lines[1:])
            
            # Clean page name (filename without folder or extension)
            page_name = os.path.splitext(os.path.basename(file_name))[0]
            
            # Add document to list
            documents.append({
                "filename": file_name,
                "page_name": page_name,
                "url": url,
                "content": page_content
            })

# Convert to DataFrame
df = pd.DataFrame(documents)

# Quick sanity check
print(f"Loaded {len(df)} markdown files")
print(df.head())
print(df['content'].str.len().describe())

Loaded 9560 markdown files
                              filename                         page_name  \
0  d680e8a854a7cbad6d490c445cba2eba.md  d680e8a854a7cbad6d490c445cba2eba   
1  42f623b3bad309d5d6619d450af47d40.md  42f623b3bad309d5d6619d450af47d40   
2  cbcc2bf9adc75ea7fce66bd4eb246203.md  cbcc2bf9adc75ea7fce66bd4eb246203   
3  717c9560db9ebed56c22664925c4c1e6.md  717c9560db9ebed56c22664925c4c1e6   
4  d8a2a1a177f439f2ca185e084642f713.md  d8a2a1a177f439f2ca185e084642f713   

                                                 url  \
0                           https://www.fordham.edu/   
1                   https://www.fordham.edu/research   
2                       https://www.fordham.edu/ccel   
3  https://www.fordham.edu/fordham-college-at-lin...   
4                  https://www.fordham.edu/academics   

                                             content  
0  \n## Doing Good That Becomes Greater As The Je...  
1  \nUncovering the Limitless Possibilities of Sc...  
2  \n# Center 

In [ ]:
# YOUR CODE HERE
# Chunking using paragraphs

enc = tiktoken.get_encoding("cl100k_base")

def fast_chunk_documents(df, max_tokens=300):
    all_chunks = []
    
    # Using a list comprehension/iterator approach is faster than while-loops on raw strings
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing Fordham Pages"):
        content = str(row['content'])
        # Pre-split by paragraphs to reduce search space
        paragraphs = content.split('\n\n')
        
        current_chunk = []
        current_tokens = 0
        
        for para in paragraphs:
            para = para.strip()
            if not para: continue
            
            para_tokens = len(enc.encode(para))
            
            # If a single paragraph is giant, force-split
            if para_tokens > max_tokens:
                # Flush existing chunk
                if current_chunk:
                    all_chunks.append({
                        "url": row['url'],
                        "page_name": row['page_name'],
                        "chunk_content": "\n\n".join(current_chunk),
                        "token_count": current_tokens
                    })
                # Split giant paragraph into smaller bit
                words = para.split()
                for i in range(0, len(words), 100): # ~130 tokens per 100 words
                    sub_para = " ".join(words[i:i+100])
                    all_chunks.append({
                        "url": row['url'],
                        "page_name": row['page_name'],
                        "chunk_content": sub_para,
                        "token_count": len(enc.encode(sub_para))
                    })
                current_chunk = []
                current_tokens = 0
                continue

            # Regular grouping
            if current_tokens + para_tokens > max_tokens:
                all_chunks.append({
                    "url": row['url'],
                    "page_name": row['page_name'],
                    "chunk_content": "\n\n".join(current_chunk),
                    "token_count": current_tokens
                })
                current_chunk = [para]
                current_tokens = para_tokens
            else:
                current_chunk.append(para)
                current_tokens += para_tokens
                
        # Catch the last chunk of the file
        if current_chunk:
            all_chunks.append({
                "url": row['url'],
                "page_name": row['page_name'],
                "chunk_content": "\n\n".join(current_chunk),
                "token_count": current_tokens
            })
                
    return pd.DataFrame(all_chunks)

# 300 tokens is roughly 1200-1500 characters
chunks_df = fast_chunk_documents(df, max_tokens=300)
print(f"Created {len(chunks_df)} chunks from {len(df)} documents.")

Processing Fordham Pages: 100%|██████████| 9560/9560 [00:15<00:00, 600.19it/s]


Created 41957 chunks from 9560 documents.


In [47]:
chunks_df.head()

,url,page_name,chunk_content,token_count
0,https://www.fordham.edu/,d680e8a854a7cbad6d490c445cba2eba,## Doing Good That Becomes Greater As The Jesu...,269
1,https://www.fordham.edu/,d680e8a854a7cbad6d490c445cba2eba,Fordham University is an Equal Opportunity Emp...,96
2,https://www.fordham.edu/research,42f623b3bad309d5d6619d450af47d40,Uncovering the Limitless Possibilities of Scie...,285
3,https://www.fordham.edu/research,42f623b3bad309d5d6619d450af47d40,A new generation of students at the Bronx Jewi...,267
4,https://www.fordham.edu/ccel,cbcc2bf9adc75ea7fce66bd4eb246203,# Center for Community Engaged Learning\n\n###...,202


---

## Step 2: Embed the Chunks

Turn each chunk into a vector so you can search over them. You can use a local model or an API model — your choice.

Once you've created your embeddings, **save them somewhere** so you (and I) don't have to redo this step. Save the chunk metadata too (text, source URL, etc.).

In [6]:
# YOUR CODE HERE
# Control the speed: 20 simultaneous requests is usually safe for Tier 1 OpenAI accounts
# Gemini-enhanced
# Using a Semaphore to limit 'concurrency' (how many tasks run at once).
# 1. Rate Limiting: OpenAI Tier 1 accounts have strict Requests Per Minute (RPM) limits.
#    Firing 30k requests instantly would get us blocked.
# 2. Memory Management: Every 'pending' request takes up a bit of RAM. 
#    A semaphore keeps our memory footprint flat by only allowing 20 active tasks.
# 3. Connection Stability: It prevents 'Connection Reset' errors by not 
#    overwhelming your local machine's available network ports.

semaphore = asyncio.Semaphore(20) 

async def embed_batch(texts, model="openai/text-embedding-3-small"):
    async with semaphore:
        try:
            response = await litellm.aembedding(
                model=model,
                input=texts
            )
            return [d['embedding'] for d in response['data']]
        except Exception as e:
            print(f"Error embedding batch: {e}")
            return [None] * len(texts)

async def run_full_embedding(chunks_df, batch_size=100):
    texts = chunks_df['chunk_content'].tolist()
    all_embeddings = []
    
    # tqdm progress bar for async
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding Progress"):
        batch = texts[i : i + batch_size]
        results = await embed_batch(batch)
        all_embeddings.extend(results)
        
        # Checkpoint every 2000 chunks
        if i > 0 and i % 2000 == 0:
            current_progress = np.array([e for e in all_embeddings if e is not None])
            np.save(f'temp/checkpoint_{i}.npy', current_progress)
            
    return all_embeddings

# --- Execute ---
# Note: Use 'await' if in a notebook, or 'asyncio.run()' in a script
raw_embeddings = await run_full_embedding(chunks_df)

Embedding Progress: 100%|██████████| 420/420 [20:23<00:00,  2.91s/it]


In [ ]:
#Save embeddings
# Extract raw results and filter out any None values (from API timeouts/errors)
openai_embeddings = np.array([e for e in raw_embeddings if e is not None])

# Normalize the embeddings for fast dot-product search
# Added a tiny epsilon (1e-9) to avoid division by zero on empty chunks
norms = np.linalg.norm(openai_embeddings, axis=1, keepdims=True)
normalized_embeddings = openai_embeddings / (norms + 1e-9)

# Save embeddings
np.save('temp/fordham_norm_embeddings.npy', normalized_embeddings)

# gemini-enhanced: Save metadata (ensure the index matches your filtered embeddings)
# If any chunks were skipped (Nones), make sure to filter the dataframe too:
# chunks_df = chunks_df.iloc[[i for i, e in enumerate(raw_embeddings) if e is not None]]
chunks_df.to_parquet('temp/fordham_metadata.parquet')

print(f"Success! {len(openai_embeddings)} embeddings saved to '/temp/'.")

Success! 41957 embeddings saved to '/temp/'.


In [2]:
#Load embeddings
print(f"Loading embeddings and metadata...")
normalized_embeddings = np.load('temp/fordham_norm_embeddings.npy')
# Load metadata
metadata_df = pd.read_parquet('temp/fordham_metadata.parquet')

print("Embeddings loaded into memory.")

Loading embeddings and metadata...


Embeddings loaded into memory.


---

## Step 3: Retrieve

Build the **R** in RAG. Write a function that takes a question and returns the most relevant chunks. You can use semantic search, BM25, hybrid — whatever you think works best.

Test it on a few questions and eyeball whether the results make sense.

In [5]:
# Hybrid retrieval: lexical (BM25) + semantic (embeddings)
from helpers import build_index, score_bm25, normalize_scores

# --- One-time BM25 index build (over all chunks) ---
# Assumes `metadata_df` has a `chunk_content` column aligned with `normalized_embeddings`.
bm25_docs = metadata_df["chunk_content"].astype(str).tolist()
bm25_index, bm25_doc_lengths = build_index(bm25_docs)
num_docs = len(bm25_docs)

assert normalized_embeddings.shape[0] == num_docs, (
    "Embeddings and metadata must have the same number of rows. "
    "If you filtered out failed embeddings, filter metadata_df the same way."
)

def _embed_query(query: str, model: str = "openai/text-embedding-3-small") -> np.ndarray:
    """Embed a single query with the same model used for chunks and L2-normalize it."""
    resp = litellm.embedding(model=model, input=query)
    vec = np.array(resp["data"][0]["embedding"], dtype=np.float32)
    vec = vec / (np.linalg.norm(vec) + 1e-9)
    return vec

def retrieve_hybrid(question: str, k: int = 10, alpha: float = 0.6) -> pd.DataFrame:
    """Hybrid retrieval using a weighted combination of semantic and BM25 scores.

    alpha controls the weight on semantic similarity (0–1). For example:
    - alpha=0.6: slightly favor semantic
    - alpha=0.5: equal weight
    """
    # --- Semantic scores ---
    q_vec = _embed_query(question)
    # normalized_embeddings is already L2-normalized, so dot = cosine similarity
    semantic_scores = normalized_embeddings @ q_vec

    # --- BM25 scores ---
    bm25_scores = score_bm25(question, bm25_index, num_docs, bm25_doc_lengths)

    # --- Normalize and fuse ---
    sem_norm = normalize_scores(semantic_scores)
    bm25_norm = normalize_scores(bm25_scores)
    hybrid_scores = alpha * sem_norm + (1.0 - alpha) * bm25_norm

    # --- Select top-k ---
    top_idx = np.argsort(-hybrid_scores)[:k]
    results = metadata_df.iloc[top_idx].copy()
    results["semantic_score"] = semantic_scores[top_idx]
    results["bm25_score"] = bm25_scores[top_idx]
    results["hybrid_score"] = hybrid_scores[top_idx]
    results["rank"] = range(1, len(results) + 1)
    return results

# Quick sanity check on a few questions
test_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid?",
    "Where is Fordham's campus?",
]
for q in test_questions:
    print("\nQUESTION:", q)
    hits = retrieve_hybrid(q, k=5)
    for _, row in hits.iterrows():
        snippet = row["chunk_content"][:140].replace("\n", " ")
        print(f"  [rank={row['rank']}] hybrid={row['hybrid_score']:.3f} | {snippet}...")

#Cursor generated



QUESTION: What programs does the Gabelli School of Business offer?
  [rank=1] hybrid=0.967 | # Gabelli Graduate Academic Programs  ![Academic Programs](/media/home/schools/gabelli-school-of-business/gsb_academic_programs.jpg)  No mat...
  [rank=2] hybrid=0.937 | The Graduate School of Social Service, the Graduate School of Religion and Religious Education the School of Professional and Continuing Stu...
  [rank=3] hybrid=0.932 | [Gabelli School of Business](/gabelli-school-of-business/academic-programs-and-admissions/undergraduate-programs/)offers distinct programs a...
  [rank=4] hybrid=0.932 | [Gabelli School of Business](/gabelli-school-of-business/academic-programs-and-admissions/undergraduate-programs/)offers distinct programs a...
  [rank=5] hybrid=0.932 | [Gabelli School of Business](/gabelli-school-of-business/academic-programs-and-admissions/undergraduate-programs/)offers distinct programs a...

QUESTION: How do I apply for financial aid?
  [rank=1] hybrid=0.923 | All complet

In [6]:
# Quick sanity check on a few questions
test_questions = [
    "Whos is Apostolos Filippas?",
    "Who is Sertan Kabadayi?",
    "Where is Fordham's campus?",
]
for q in test_questions:
    print("\nQUESTION:", q)
    hits = retrieve_hybrid(q, k=5)
    for _, row in hits.iterrows():
        snippet = row["chunk_content"][:140].replace("\n", " ")
        print(f"  [rank={row['rank']}] hybrid={row['hybrid_score']:.3f} | {snippet}...")



QUESTION: Whos is Apostolos Filippas?
  [rank=1] hybrid=1.000 | # Apostolos Filippas  ![Apostolos Filippas](/media/home/schools/gabelli-school-of-business/gsb-Apostolos-Filippas.jpg)  Assistant Professor ...
  [rank=2] hybrid=0.916 | Robert Chiang, Ph.D., earned his doctoral degree in information systems from the University of Washington. Prior to joining the Gabelli Scho...
  [rank=3] hybrid=0.895 | Dawit Demissie, Ph.D., is currently a clinical professor at Fordham University’s, Gabelli School of Business, recognized for his dedication ...
  [rank=4] hybrid=0.883 | **Teaching Courses: **Web Analytics, E-Commerce  Apostolos Filippas, Ph.D., is an economist working on market design and the economics of te...
  [rank=5] hybrid=0.835 | - - Filippas, Apostolos, Srikanth Jagabathula, and Arun Sundararajan (2023), “The Limits of Centralized Pricing in Online Marketplaces and t...

QUESTION: Who is Sertan Kabadayi?
  [rank=1] hybrid=0.992 | # Sertan Kabadayi  ![Sertan Kabadayi](/media/home/

---

## Step 4: Generate

Build the **G** in RAG. Write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.

Think about: how to structure the prompt, what the LLM should do when the context doesn't contain the answer, and which model to use.

In [7]:
import textwrap

def build_rag_prompt(question: str, retrieved_chunks: pd.DataFrame, max_chars: int = 6000) -> tuple[str, str]:
    """Construct system + user messages for RAG generation.

    - retrieved_chunks: output of retrieve_hybrid (must have 'chunk_content', 'url', 'page_name').
    - max_chars: hard cap on total context length to keep prompts within model limits.
    """
    # Join chunk texts with clear separators, optionally including source URLs
    context_pieces = []
    for _, row in retrieved_chunks.iterrows():
        source = row.get("url", "")
        header = f"[SOURCE: {source}]" if source else ""
        context_pieces.append(f"{header}\n{row['chunk_content']}")

    context = "\n\n---\n\n".join(context_pieces)
    if len(context) > max_chars:
        context = context[:max_chars]

    system_msg = (
        # "You are a helpful assistant that answers questions about Fordham University. "
        # "Use ONLY the provided context when answering. If the answer is not in the context, "
        # "say that you don't know and, if possible, suggest where on fordham.edu the user might look."
        # Enhanced context
        "You are a helpful, professional assistant for Fordham University. "
        "Your task is to answer user questions using ONLY the provided context. "
        "\n\nGUIDELINES:"
        "\n- If the answer is not in the context, say: 'I'm sorry, I don't have enough information to answer that based on the university records.' Do not make up facts."
        "\n- Cite your sources by mentioning the page name where the information was found."
        "\n- Keep your tone friendly and academic."
    )

    user_msg = f"""Answer the user's question using ONLY the context below.

CONTEXT:
{context}

QUESTION:
{question}

If the context does not contain the answer, explicitly say you don't know.
"""

    return system_msg, user_msg


def generate_answer(question: str, retrieved_chunks: pd.DataFrame, model: str = "gpt-4o-mini") -> str:
    """Run the G step: take retrieved chunks + question and call an LLM.

    This function is side-effect free and suitable for reuse in a Streamlit app.
    """
    system_msg, user_msg = build_rag_prompt(question, retrieved_chunks)

    resp = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.2,
    )

    return resp["choices"][0]["message"]["content"].strip()


# Quick test of R + G together using hybrid retrieval
demo_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid?",
    "Whos is Apostolos Filippas?"
]
for q in demo_questions:
    print("\n=== QUESTION ===\n", q)
    retrieved = retrieve_hybrid(q, k=5)
    answer = generate_answer(q, retrieved)
    print("\nANSWER:\n", textwrap.fill(answer, width=100))
#Cursor


=== QUESTION ===
 What programs does the Gabelli School of Business offer?

ANSWER:
 The Gabelli School of Business offers three variants of the MBA program: a full-time cohort M.B.A, a
Professional M.B.A (including part-time), and an Executive M.B.A. Additionally, it provides 13
specialized Master of Science programs tailored for specific business niches. For more details, you
can refer to the Gabelli Graduate Academic Programs page.

=== QUESTION ===
 How do I apply for financial aid?

ANSWER:
 To apply for financial aid at Fordham University, you should follow these steps:  1. Select “Yes”
when asked if you are applying for financial aid on the Fordham University Member Questions of the
Common Application. 2. Submit the CSS Profile (school code: 2259) and any supporting documents by
the deadlines for your chosen admission plan:    - Early Action / Early Decision I: Admission
Application and CSS Profile by November 1    - Early Decision II: Admission Application and CSS
Profile by J

---

## Step 5: Wire it Together

Combine the previous steps into a single `rag(question)` function. Question in, answer out.

In [8]:
def rag(question: str,
        k: int = 8,
        alpha: float = 0.6,
        model: str = "gpt-4o-mini") -> dict:
    """End-to-end RAG pipeline: Question in, answer + sources out.

    Designed to be production-friendly:
    - Retrieval uses precomputed global state (normalized_embeddings, bm25_index, metadata_df).
    - Generation calls a single cheap chat model via litellm.
    - Returns both the answer and the top-k chunks for inspection/attribution.
    """
    # Retrieve relevant chunks with hybrid search
    retrieved = retrieve_hybrid(question, k=k, alpha=alpha)

    # Generate an answer constrained to the retrieved context
    answer = generate_answer(question, retrieved, model=model)

    # Minimal, structured response for downstream use (e.g., Streamlit, API)
    return {
        "question": question,
        "answer": answer,
        "sources": retrieved[["url", "page_name"]].drop_duplicates().to_dict(orient="records"),
        "retrieved_chunks": retrieved,
    }

demo_q = "What programs does the Gabelli School of Business offer?"
result = rag(demo_q)
print("QUESTION:\n", result["question"])
print("\nANSWER:\n", textwrap.fill(result["answer"], width=100))
print("\nSOURCES:")
for src in result["sources"]:
    print(" -", src.get("page_name"), "::", src.get("url"))


QUESTION:
 What programs does the Gabelli School of Business offer?

ANSWER:
 The Gabelli School of Business offers three variants of the MBA program: a full-time cohort M.B.A, a
Professional M.B.A (including part-time), and an Executive M.B.A. Additionally, it provides 13
specialized Master of Science programs tailored for specific business niches. For more details, you
can refer to the Gabelli Graduate Academic Programs page.

SOURCES:
 - 33de6736963a941fbac189cdc215494f :: https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions/graduate-programs/academic-programs
 - a6058b97f9c115f92f15b91eed5f52fc :: https://www.fordham.edu/resources/policies/credit-assignment-policy
 - 9d32c1499695d4d17e55ae6f9793ef5d :: https://www.fordham.edu/info/23872/transfer_credit_policies_and_procedures
 - afc1fa4b718e518210d83f2456aec91b :: https://www.fordham.edu/undergraduate-admission/apply/how-to-apply/transfer-students
 - ca798be26ee789a09746def1a2d23642 :: https://www.for

In [63]:
# Demonstrate your RAG system

demo_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid at Fordham?",
    "What is the tuition for undergraduate students?",
    "Tell me about Fordham's campus locations.",
    "What research opportunities are available for students?",
]

In [64]:
#Answers only
for q in demo_questions:
    res = rag(q)         
    print("Q:", q)
    print("A:", textwrap.fill(res["answer"], width=100))
    print("-" * 80)

Q: What programs does the Gabelli School of Business offer?
A: The Gabelli School of Business offers three variants of the MBA program: a full-time cohort M.B.A, a
Professional M.B.A (including part-time), and an Executive M.B.A. Additionally, it provides 13
specialized Master of Science programs tailored for specific business niches. (Source: Gabelli
Graduate Academic Programs)
--------------------------------------------------------------------------------
Q: How do I apply for financial aid at Fordham?
A: To apply for financial aid at Fordham, you should follow these steps:  1. Select “Yes” when asked if
you are applying for financial aid on the Fordham University Member Questions of the Common
Application. 2. Submit the CSS Profile (school code: 2259) and any supporting documents by the
deadlines for your chosen admission plan:    - Early Action / Early Decision I: Admission
Application and CSS Profile by November 1    - Early Decision II: Admission Application and CSS
Profile by J

---

## Step 6: Evaluate Your RAG System

A working RAG system is great — but how do you know it's actually *good*? You can't improve what you can't measure. In this step you'll build an evaluation framework using concepts from Lecture 6.

There are two things to evaluate in a RAG system:
- **Retrieval quality**: Are you finding the right chunks?
- **Answer quality**: Is the generated answer correct and grounded in the context?

### Build a test set

Create a test set of at least **10 question-answer pairs**. For each pair, provide the question and the expected answer (look it up in the data). Cover a range of question types — factual, procedural, about specific programs, etc.

### Evaluate retrieval

For each question, check whether the retrieved chunks actually contain relevant information. You can do this manually or automatically (e.g., use an LLM to judge relevance). Compute **context relevance** — the fraction of retrieved chunks that are actually useful.

### Evaluate answers with LLM-as-judge

Use an LLM to evaluate your system's answers on two dimensions:

1. **Faithfulness**: Does the answer only use information from the retrieved context? (No hallucination)
2. **Correctness**: Is the answer factually correct compared to the expected answer?

Use **structured outputs** (Pydantic) to get consistent scores from the judge. A starting schema is provided below — feel free to modify it.

In [65]:
from pydantic import BaseModel, Field

class RAGEvaluation(BaseModel):
    faithfulness_score: int = Field(
        ..., ge=1, le=5,
        description="1=completely hallucinated, 5=fully grounded in context"
    )
    faithfulness_reasoning: str = Field(
        ..., description="Brief explanation of the faithfulness score"
    )
    correctness_score: int = Field(
        ..., ge=1, le=5,
        description="1=completely wrong, 5=fully correct and complete"
    )
    correctness_reasoning: str = Field(
        ..., description="Brief explanation of the correctness score"
    )

In [75]:
# YOUR CODE HERE
# - Build your test set
# - Evaluate retrieval quality (context relevance)
# - Evaluate answer quality with LLM-as-judge (faithfulness + correctness)
# - Summarize your results

In [76]:
# - Build your test set
import random

# # 'chain_of_thought' makes the LLM reason before generating the question
class SyntheticQuestion(BaseModel):
    chain_of_thought: str = Field(description="Step-by-step reasoning about what makes a good question")
    question: str = Field(description="A natural, specific Fordham-related question that can answered with provided documents")
    answer: str = Field(description="The answer to the question grounded in the text")

In [84]:
# - Build your test set
# random "constraints" 
constraints = [
    # --- Prospective Graduate / Professional ---
    "Frame the question as a prospective graduate student asking about MA or MS or MBA prerequisites.",
    "Ask about the difference between full-time and part-time enrollment options for a professional degree.",
    
    # --- Researchers / Faculty ---
    "Frame the question as a researcher looking for information on internal funding or grant opportunities.",
    "Ask about specific specialized labs, research centers, or faculty-led initiatives mentioned.",
    
    # --- Employees / Staff ---
    "Frame the question as a new Fordham employee asking about benefits, parking, or HR policies.",
    "Ask about the specific administrative process for expense reports or campus ID card activation.",

    # --- Structural / Complex ---
    "The question should require synthesizing multiple facts (e.g., combining a program name with a specific campus location).",
    "Ask about a specific number, GPA threshold, or date/deadline mentioned in the document.",
    
    # --- Edge Cases ---
    "Frame a question that is slightly outside the current context to test if the model says 'Information Not Found'."
]

# Generate questions for testing

async def generate_question(doc_id: str, title: str, text: str) -> dict:
    """Generate a synthetic question for a single Fordham chunk using an LLM."""
    constraint = random.choice(constraints)
    response = await litellm.acompletion(
        model="gpt-4o-mini",   
        messages=[
            {
                "role": "user",
                "content": textwrap.dedent(f"""
                I will give you a document from Fordham University's website.
                Please generate a question that can be answered using the following text.

                Title: {title}
                Text: {text}

                Rules:
                - Your question should be natural, specific, and concise.
                - Your question should not assume that someone is reading the page, but rather that they are asking a general question about Fordham.
                - Your question must be answerable using the text that I gave you.
                - {constraint}
                - Do not reference "this page", "the document", or "the site" in your question.
                """),
            }
        ],
        response_format=SyntheticQuestion,
    )

    # Parse the JSON response into our Pydantic model
    result = SyntheticQuestion.model_validate_json(response.choices[0].message.content)
    return {
        "doc_id": doc_id,
        "question": result.question,
        "expected_answer": result.answer,
    }

In [78]:
# Sample N chunks from Fordham chunks_df
sample_chunks = chunks_df.sample(n=80, random_state=42)

tasks = [
    generate_question(
        doc_id=row["page_name"],           
        title=row["page_name"],
        text=row["chunk_content"],
    )
    for _, row in sample_chunks.iterrows()
]

synthetic_results = await asyncio.gather(*tasks)
synthetic_df = pd.DataFrame(synthetic_results)

In [79]:
print(f"Generated {len(synthetic_df)} synthetic questions\n")

# Show some examples with their source documents
for _, row in synthetic_df.head(5).iterrows():
    doc = chunks_df[chunks_df["page_name"] == row["doc_id"]].iloc[0]
    print(f"Q: {row['question']}")
    print(f"   Source: {doc['url']}")
    print(f"   Answer: {row['expected_answer']}")
    print()

Generated 80 synthetic questions

Q: What is Rodrigo Recinos' area of focus in theology, and where is he currently studying?
   Source: https://www.fordham.edu/academics/departments/theology/graduate-students/our-current-graduate-students/rodrigo-recinos
   Answer: Rodrigo Recinos is focused on liberation theology and is currently a doctoral student at Fordham University.

Q: What are some of the anthropology courses offered at Fordham University and who teaches them?
   Source: https://www.fordham.edu/academics/research/faculty-research/fordham-africanist-group/courses
   Answer: The anthropology courses offered include ANTH 3354 - Race, Identity, and Globalization taught by Huda Gerard-Seif, and ANTH 4490 - Anthropology of Political Violence also taught by Huda Gerard-Seif.

Q: What skills do Fordham graduates develop that are applicable to their careers?
   Source: https://www.fordham.edu/fordham-stories/campus-and-city-life/they-started-a-cluband-kickstarted-their-careers
   Answer

In [85]:
# Compute context relevance
def compute_context_relevance(
    retrieved_chunks: pd.DataFrame,
    expected_answer: str
) -> float:
    """
    Simple automatic relevance metric:
    fraction of chunks containing keywords from expected answer.
    """
    expected_terms = set(expected_answer.lower().split())
    relevant = 0

    for chunk in retrieved_chunks["chunk_content"]:
        chunk_terms = set(chunk.lower().split())
        if len(expected_terms.intersection(chunk_terms)) > 2:
            relevant += 1

    return relevant / len(retrieved_chunks)

In [109]:

async def judge_answer(example: dict, k: int = 8, alpha: float = 0.6) -> RAGEvaluation:
    """Use an LLM to score one RAG answer on faithfulness + correctness."""
    # Run our RAG system to get answer + context
    rag_result = rag(example["question"], k=k, alpha=alpha)
    context_chunks = rag_result["retrieved_chunks"]["chunk_content"].tolist()
    context_text = "\n\n---\n\n".join(context_chunks)

    prompt = textwrap.dedent(f"""
    You are an evaluation assistant for a Fordham University RAG system.

    Evaluate the *student answer* below on two axes:
    1) Faithfulness: Does it rely ONLY on the provided context? (No hallucinated facts.)
    2) Correctness: Is it factually correct compared to the expected answer (allowing paraphrase)?

    CONTEXT:
    {context_text}

    QUESTION:
    {example["question"]}

    EXPECTED_ANSWER (from instructor or docs):
    {example["expected_answer"]}

    STUDENT_ANSWER (from RAG system):
    {rag_result["answer"]}
    """)

    resp = await litellm.acompletion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format=RAGEvaluation,
        temperature=0.0,
    )

    scores = RAGEvaluation.model_validate_json(
        resp.choices[0].message.content
    )

    relevance = compute_context_relevance(
        rag_result["retrieved_chunks"],
        example["expected_answer"]
    )

    return {
        "question": example["question"],
        "faithfulness_score": scores.faithfulness_score,
        "correctness_score": scores.correctness_score,
        "context_relevance": relevance,
    }



# LLM Judge
async def run_llm_judge_eval(test_set: list[dict]) -> pd.DataFrame:
    """Evaluate a list of QA examples and store actual RAG answers."""
    rows = []
    for ex in test_set:
        scores = await judge_answer(ex)
        rows.append(
            {
                "question": ex["question"],
                "expected_answer": ex["expected_answer"],
                "faithfulness_score": scores["faithfulness_score"],
                "correctness_score": scores["correctness_score"],
            }
        )

    return pd.DataFrame(rows)

In [111]:
# Run the evaluation

test_examples = synthetic_df.head(8).to_dict(orient="records")

judge_df = await run_llm_judge_eval(test_examples)

display(judge_df)

print("\nAverage scores:")
print(
    judge_df[["faithfulness_score", "correctness_score"]]
    .mean()
    .round(2)
    .to_string()
)

,question,expected_answer,faithfulness_score,correctness_score
0,What is Rodrigo Recinos' area of focus in theo...,Rodrigo Recinos is focused on liberation theol...,5,4
1,What are some of the anthropology courses offe...,The anthropology courses offered include ANTH ...,5,5
2,What skills do Fordham graduates develop that ...,Fordham graduates develop skills such as clien...,5,5
3,What courses does Serge Reda teach at Fordham ...,Serge Reda teaches Real Estate Development Pro...,5,5
4,How can prospective graduate students learn ab...,Prospective graduate students can connect with...,5,1
5,What resources does Fordham University offer t...,Fordham University provides various resources ...,5,5
6,What role does the executive director of the R...,The executive director of the Responsible Busi...,5,5
7,What are the main goals of the course focused ...,The main goals of the course are to enjoy Aust...,5,5



Average scores:
faithfulness_score    5.00
correctness_score     4.38


---

## Step 7: Build a Streamlit App

Your RAG system lives inside a notebook — that's great for development, but nobody is going to use a Jupyter notebook to ask questions about Fordham. Turn it into a web app using [Streamlit](https://docs.streamlit.io/).

Create a `.py` file (e.g., `scripts/fordham_rag_app.py`) that:
1. Lets the user type a question about Fordham
2. Runs your RAG pipeline
3. Displays the answer and the source pages used

**Getting started:**
- Install: `uv pip install streamlit`
- Run: `streamlit run scripts/fordham_rag_app.py`

**Tip**: Use `@st.cache_resource` to avoid reloading embeddings on every interaction.

**Include a screenshot of your working app below.**

*Paste your screenshot here*
![alt text](<Fordham RAG Assistant_20260225.png>)

---

## Step 8: How to Run Your System

Fill in the details below so that I can run and test your RAG system.

| Item | Your Answer |
|------|-------------|
| **Embedding model used** |openai/text-embedding-3-small |
| **LLM used for generation** |gpt-4o-mini |
| **LLM used for evaluation (judge)** | gpt-4o-mini|
| **Saved artifacts** | temp/fordham_norm_embeddings.npy, temp/fordham_metadata.parquet, images/fordhamlogo.svg|
| **How to start the Streamlit app** | (e.g., `streamlit run scripts/fordham_rag_app.py`) |
| **Any API keys or env vars needed** | (e.g., `OPENAI_API_KEY` in `.env`) |
| **Anything else I should know** | |

---

## Bonus: Experiment and Improve

Now that you have a working RAG system *and* a way to measure its quality, try to improve it. Use your evaluation framework to measure the impact of changes.

Ideas: different chunk sizes, different embedding models, hybrid search, better prompts, reranking, query rewriting. Document what you tried and show before/after evaluation scores.

In [114]:
# YOUR CODE HERE



---

## Git Submission

- [ ] Create a new branch called `homework-5`
- [ ] Commit your work (notebook + Streamlit app + saved artifacts in `temp/`)
- [ ] Push to GitHub
- [ ] Create a Pull Request and merge to main
- [ ] Submit the `.ipynb` file on Blackboard